In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
from scipy.integrate import solve_ivp
from scipy.linalg import solve,schur
from utility import orbit, BrusselatorModel, optim_BrusselatorModel, call_method
import run_analysis as ra
import sys,os, argparse,time, pickle
from datetime import datetime

from pathlib import Path
from scipy.sparse.linalg import LinearOperator
from scipy.sparse.linalg import eigs
from scipy.interpolate import interp1d

from matplotlib.backends.backend_pdf import PdfPages
from joblib import Parallel, delayed
import imageio
import cProfile
import pandas as pd

In [ ]:
def run(model,n_z,orbit_method,p0,filename=None):

    epsilon = model.precision
    model.n_z = n_z
    model.p0 = p0 #Size of the dominant subspace
    model.Lap = model.Lap_mat() #Upgrade the Laplacian matrix according to the new grid size
    f = model.dydt
    Jacf = model.brusselator_jacobian
    #Initialization
    X0 = model.A + 0.1*np.sin(np.pi*(np.linspace(0, model.z_L, model.n_z)/model.z_L))
    Y0 = model.B/model.A + 0.1*np.sin(np.pi*(np.linspace(0, model.z_L, model.n_z)/model.z_L))
    y0 = np.concatenate([X0[1:-1],Y0[1:-1]])
    #We integrate sufficiently the equation to find a good starting point
    phi_t = solve_ivp(fun=f,t_span=[0.0, 16*model.T_ini],
                t_eval=[16*model.T_ini],
                # dense_output=True,
                y0=y0, method=model.method,
                jac=Jacf,
                **{"rtol": 1e-5,"atol":1e-7}
                )
    
    y_T = phi_t.y[:,-1] #Using phi(y0,T0) as a starting point
    orbit_finder = orbit(f,y_T,model.T_ini, Jacf,2, solve_ivp, model.method, 10000,model.max_iter, epsilon)

    V_0 = np.eye(len(y_T))[:,:p0+model.pe]#Initial guess of the subspace
    #The arguments to pass to the orbit_finder method
    args_func = {
    "y0": y_T,
    "T_0": model.T_ini,
    "Max_iter": model.max_iter,
    "epsilon": epsilon,
    "subsp_iter": model.subsp_iter,
    "l": model.picard_iter,
    "Ve_0": V_0,
    "p0": p0,
    "pe": model.pe,
    "rho": model.rho,
    "phase_cond": 2,
    "l": model.picard_iter,
    "full_sub_iter": model.full_sub_iter,  # Use the full subspace iteration if True for the subspace iteration with projection
    }
    method_to_call= getattr(orbit_finder, orbit_method)

    start = time.time()
    k, T_by_iter, y_by_iter, Norm_B, Abs_Err, Rel_Err, converged = call_method(method_to_call, **args_func)
    end = time.time()
    total_time = end - start
               
    results = dict(
        orbit_method = orbit_method,
        solv_method = model.method,
        nz = n_z,
        p0 = p0,
        pe=model.pe,
        sub_sp_iter = model.subsp_iter,
        full_sub_iter = model.full_sub_iter,
        rho = model.rho,
        n_iter = k,
        abs_err = Abs_Err[k],
        rel_err = Rel_Err[k],
        norm_B = Norm_B[k],
        converged = converged,
        comput_time = total_time,
        T_star = T_by_iter[k],
        
    )
    
    return results, k, T_by_iter, y_by_iter, Norm_B, Abs_Err, Rel_Err, converged

In [ ]:
if __name__ == "__main__":
    param_file = "./RK45_bruss_dflt_params.in"  # JSON file containing model parameters
    param_file_2 = "./RK45_notfull_subiter_bruss.in"
    model = BrusselatorModel(param_file)
    model_optim = optim_BrusselatorModel(param_file)
    model_optim2 = optim_BrusselatorModel(param_file_2)
    print("Loaded parameters:", model_optim2.n_z)



    nz=256
    Res2, k2, T_by_iter2, y_by_iter2, Norm_B2, Abs_Err2, Rel_Err2, converged2 = run(model_optim2, nz, "Newton_Picard_sub_proj", p0=5)
    # Res, k, T_by_iter, y_by_iter, Norm_B, Abs_Err, Rel_Err, converged = run(model_optim, nz, "Newton_orbit", p0=5)

In [ ]:
#plot convergence results and mass evolution using subplots
fontsize = 20
fig = plt.figure(figsize=(10,6))
# plt = fig.add_subplot(211)
# plt.semilogy(range(k+1), Norm_B[:k+1],'*-', label='Norm of B')
# plt.semilogy(range(k+1), Abs_Err[:k+1],'*-', label='Absolute Error')
plt.semilogy(range(k+1), Rel_Err[:k+1], '*-',label='Newton')
plt.xlabel('Iteration', fontsize=fontsize)
plt.ylabel('Error', fontsize=fontsize)
plt.title('Relative error vs iteration for nz={}'.format(nz), fontsize=fontsize)
# plt.legend(fontsize=14)
# plt.grid() 
# ax2 = fig.add_subplot(212)

# plt.semilogy(range(k2+1), Norm_B2[:k2+1],'*-', label='Norm of B')
# plt.semilogy(range(k2+1), Abs_Err2[:k2+1],'*-', label='Absolute Error')
plt.semilogy(range(k2+1), Rel_Err2[:k2+1], '*-',label='NP')
# ax2.set_xlabel('Iteration')
# ax2.set_ylabel('Error')
# ax2.set_title('Convergence of the Newton Method with Mass Conservation')
plt.legend(fontsize=14)
plt.grid()
# ax2.grid()  
plt.savefig(f'./Results/RK45_bruss_dflt_params.in/precision_vs_nz_{nz}.png')
plt.show()  
